## Replay checkpoint

In [ ]:
import os
import jax
import mujoco
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from omegaconf import OmegaConf
from pathlib import Path
from jax.scipy.spatial.transform import Rotation
import jax.numpy as jnp
from orbax import checkpoint as ocp
from typing import Any, Dict, Optional
import mediapy as media
import warnings
import numpy as np
import matplotlib.pyplot as plt
import tensegrity_playground
from mujoco_playground import registry
# import scienceplots

# plt.style.use(["science", "grid"])

warnings.filterwarnings(
    "ignore",
    message="Couldn't find sharding info under RestoreArgs.*",
    category=UserWarning,
)

In [ ]:
#test
def get_ppo_inference_fn(
    obs_size,
    act_size,
    normalize_obs: bool,
    network_factory_kwargs,
    variables,  # (normalizer_params, policy_variables)
):
    normalizer_params, policy_variables = variables

    def make_inference_fn(
        observation_size: int,
        action_size: int,
        normalize_observations: bool = True,
        network_factory_kwargs: dict | None = None,
    ):
        normalize = (lambda x, y: x)
        if normalize_observations:
            normalize = running_statistics.normalize

        ppo_net = ppo_networks.make_ppo_networks(
            observation_size,
            action_size,
            preprocess_observations_fn=normalize,
            **(network_factory_kwargs or {}),
        )
        make_policy = ppo_networks.make_inference_fn(ppo_net)
        return make_policy

    make_policy = make_inference_fn(
        obs_size,
        act_size,
        normalize_obs,
        network_factory_kwargs,
    )

    # 包一层，把 key 作为输入参数之一
    def policy_apply(obs, key):
        apply_fn = make_policy((normalizer_params, policy_variables), deterministic=True)
        return apply_fn(obs, key)  # 关键：传入 key_sample

    # JIT 并暴露需要的签名
    jit_inference_fn = jax.jit(policy_apply)
    return jit_inference_fn

def get_body_world_linvel(model: mujoco.MjModel, data: mujoco.MjData, body_name: str):
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
    # data.cvel[body_id] 是 6D 空间速度（世界系）：[ang_vel, lin_vel]
    v6 = np.array(data.cvel[body_id], dtype=np.float64)  # 确保是 numpy
    return v6[3:]  # (vx, vy, vz)

In [ ]:
# ========= 加载 =========
path = "/media/di/4441-E469/cluster_tmp/tens3_8000_10_rk4-2700048/250929002525-tensegrityquadrupedwalk-ppo-1/"
checkpoint = "ckpt_72417280"
path = Path(path)
cfg_path = path / "metadata.yaml"
ckpt_path = path / "checkpoints" / checkpoint

config = OmegaConf.load(cfg_path)
cfg_overrides = registry.get_default_config(config.environment_id)
env = registry.load(config.environment_id, config_overrides=cfg_overrides)

jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

orbax_checkpointer = ocp.PyTreeCheckpointer()
normalizer_raw, policy_variables, value_variables = orbax_checkpointer.restore(ckpt_path, item=None)

print("Successfully loaded checkpoint with 3 components")

# 将 normalizer 参数恢复为 RunningStatisticsState（如果 normalize() 需要该类型）
normalizer_state = running_statistics.RunningStatisticsState(**normalizer_raw)

# print("Network factory kwargs:", config.agent.network_factory)
# print("All config keys:", list(config.keys()))
# print("Agent config keys:", list(config.agent.keys()) if hasattr(config, 'agent') else "No agent config")

# 构建推理函数
jit_inference_fn = get_ppo_inference_fn(
    env.observation_size,
    env.action_size,
    normalize_obs=True,
    network_factory_kwargs=config.agent.network_factory,
    variables=(normalizer_state, policy_variables),
)

In [ ]:
# import csv
# output_file = "go1_fixed_spine_euler.csv"

# with open(output_file, mode="w", newline="") as file:
#     writer = csv.writer(file)
#     writer.writerow(["Step", "vx (m/s)", "vy (m/s)", "vz (m/s)", "speed_xy (m/s)"])  # Header row

rng = jax.random.key(1)
rng, rng_init = jax.random.split(rng)
state = jit_reset(rng_init)
rollout = [state]
ctrl_hist = []

episode_length = 0
for i in range(1000):
    rng, act_rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)

    print(f"Observation at step {i}: {state.obs}")
    state = jit_step(state, ctrl)

    vx, vy, vz = get_body_world_linvel(env.mj_model, state.data, "vertebrae_0")
    speed_xy = float(np.hypot(vx, vy))
    print(f"vx={vx:.3f} m/s, vy={vy:.3f} m/s, speed_xy={speed_xy:.3f} m/s")

    # with open(output_file, mode="a", newline="") as file:
    #     writer = csv.writer(file)
    #     writer.writerow([i, vx, vy, vz, speed_xy])
        
    if state.done:        
        state = jit_reset(rng_init)
        episode_length = 0
        
    rollout.append(state)
    ctrl_hist.append(ctrl)

In [ ]:
render_every = 1
fps = 1.0 / env.dt / render_every

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = False

camera_names = [
    mujoco.mj_id2name(env.mj_model, mujoco.mjtObj.mjOBJ_CAMERA, i)
    for i in range(env.mj_model.ncam)
]
camera_names.append(0)

for camera in camera_names:
    frames = env.render(
        rollout[::render_every],
        height=480,
        width=640,
        camera=camera,
        scene_option=scene_option,
    )
    media.show_video(frames, fps=fps)

In [ ]:
# rng = jax.random.key(1)
# rng, rng_init = jax.random.split(rng)
# state = jit_reset(rng_init)
# rollout = [state]
# ctrl_hist = []

# episode_length = 0
# for i in range(1000):
#     rng, act_rng = jax.random.split(rng)
    
#     # 修改观察：将 privileged_state 设为全零
#     modified_obs = {
#         "state": state.obs["state"],
#         "privileged_state": jnp.zeros_like(state.obs["privileged_state"])  # 全零
#     }
    
#     ctrl, _ = jit_inference_fn(modified_obs, act_rng)

#     # print(f"Observation at step {i}: state shape={state.obs['state'].shape}, privileged_state=ZEROS")
#     state = jit_step(state, ctrl)

#     vx, vy, vz = get_body_world_linvel(env.mj_model, state.data, "vertebrae_0")
#     speed_xy = float(np.hypot(vx, vy))
#     print(f"vx={vx:.3f} m/s, vy={vy:.3f} m/s, speed_xy={speed_xy:.3f} m/s")

#     # with open(output_file, mode="a", newline="") as file:
#     #     writer = csv.writer(file)
#     #     writer.writerow([i, vx, vy, vz, speed_xy])
        
#     if state.done:        
#         state = jit_reset(rng_init)
#         episode_length = 0
        
#     rollout.append(state)
#     ctrl_hist.append(ctrl)